# Baseline Replication: Additive Prompt Tuning (ICCV 2025) — M0 Cosine vs M1 KAC

## 📌 Experiment Overview
- **Target Model:** Additive Prompt Tuning (APT) (Chen et al., ICCV 2025)
- **Evaluation Dataset:** CUB-200-2011 (200 fine-grained bird species, 10 tasks x 20 classes)
- **Backbone:** Pre-trained Vision Transformer (`ViT-B/16` frozen, 768-dim)
- **Objective:** Strict scientific replication of original baseline (M0) versus naive KAC classification (M1).

## 🔬 Head Configurations (M0 vs M1)
| Mode | Classifier Head | Formulation | Purpose |
| :--- | :--- | :--- | :--- |
| **M0 (Paper Baseline)** | Linear / Cosine Head | $y = \text{Linear}(f)$ or $\text{Norm}(f) \mathbf{W}^\top$ | Replicate original Additive Prompt Tuning (APT) paper performance |
| **M1 (Naive KAC)** | Kolmogorov-Arnold Head | $y = \text{SplineLinear}(\text{RBF}(\text{LayerNorm}(f)))$ | Test naive non-linear spline classification hypothesis |

## ⚙️ Experimental Controls (Apples-to-Apples)
- **Fixed Seeds:** [0] (with multi-split: 5T, 10T, 20T, 40T)
- **Optimizer:** Adam, standard learning rates
- **Batch Size & Epochs:** `batch_size=128`, `epochs=20`
- **Changelog:** Configured strictly for pure M0 (CosineHead) and M1 (KACHead). Removed M3 Hybrid to ensure clean baseline replication.


## 1. Setup

In [ ]:
import os, sys, json, time, math, random, subprocess, warnings
warnings.filterwarnings("ignore")
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image

print("torch", torch.__version__, "| timm", timm.__version__)
assert torch.cuda.is_available(), "Chưa bật GPU: Settings → Accelerator → GPU T4 x2"
DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


## 2. Config
`head_lr['kac']` là siêu tham số nên sweep đầu tiên: với Adam, lr 0.02 của APT thường quá lớn cho head KAC (12 288 input/lớp).

In [ ]:
CFG = dict(
    seeds=[0],                 # báo cáo: [1, 2, 3] như APT
    n_classes=200,
    n_tasks=10,                # 10 task x 20 lớp
    max_tasks=None,            # None = chạy hết; số nguyên để chạy thử vài task
    epochs=20,                 # APT paper: 18 epoch/task cho CUB (script gốc dùng 25)
    batch_size=128,
    prompt_lr=0.02,            # lr của APT cho CUB200
    head_lr={"linear": 0.02, "kac": 1e-3, "hybrid_cos": 0.02, "hybrid_kac": 1e-3},
    ema_coeff=0.7,             # hệ số merge prompt (PPF) của APT cho CUB
    clf_types=["linear", "kac"],
    tau=2.0,
    alpha=1.5,
    kac_num_grids=4,          # code KAC dùng 16 (paper ghi 4)
    kac_grid_range=(-2.0, 2.0),
    use_amp=True,              # fp16 cho backbone, head luôn fp32
    num_workers=4,
    max_train_batches=None,    # debug: giới hạn số batch mỗi epoch
    # ===== ĐƯỜNG DẪN DATASET (tự điền) =====
    # Mỗi thư mục chứa 200 thư mục lớp, vd. train/001.Black_footed_Albatross/*.jpg
    train_dir="/kaggle/input/YOUR-DATASET/train",
    test_dir="/kaggle/input/YOUR-DATASET/test",
    out_dir="/kaggle/working/outputs",
    timm_name="vit_base_patch16_224.augreg_in21k_ft_in1k",  # = vit_base_patch16_224 trong timm 0.4.12 mà APT/KAC dùng
)

DEBUG = False   # True: chạy siêu nhanh 2 task x 1 epoch để kiểm tra pipeline
if DEBUG:
    CFG.update(seeds=[0], max_tasks=2, epochs=1, max_train_batches=3)

os.makedirs(CFG["out_dir"], exist_ok=True)
print(json.dumps(CFG, indent=1))


## 3. Dữ liệu CUB-200

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def list_classes(d):
    return sorted(c for c in os.listdir(d) if os.path.isdir(os.path.join(d, c)) and not c.startswith("."))

def load_split(split_dir, classes):
    idx = {c: i for i, c in enumerate(classes)}
    items = []
    for c in classes:
        d = os.path.join(split_dir, c)
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith(IMG_EXT):
                items.append((os.path.join(d, fn), idx[c]))
    return items

TRAIN_DIR, TEST_DIR = CFG["train_dir"], CFG["test_dir"]
assert os.path.isdir(TRAIN_DIR), f"Không thấy train_dir: {TRAIN_DIR}"
assert os.path.isdir(TEST_DIR), f"Không thấy test_dir: {TEST_DIR}"

CLASSES = list_classes(TRAIN_DIR)                       # nhãn 0..199 theo thứ tự tên thư mục
assert len(CLASSES) == CFG["n_classes"], f"train_dir có {len(CLASSES)} lớp, cần {CFG['n_classes']}"
assert list_classes(TEST_DIR) == CLASSES, "Tên thư mục lớp ở train và test không khớp"

TRAIN_ITEMS = load_split(TRAIN_DIR, CLASSES)
TEST_ITEMS = load_split(TEST_DIR, CLASSES)
print(f"Train: {len(TRAIN_ITEMS)} ảnh | Test: {len(TEST_ITEMS)} ảnh | {len(CLASSES)} lớp")
print("Ví dụ:", TRAIN_ITEMS[0])


In [ ]:
# Transform giống APT/CODA-P: KHÔNG normalize (mean 0, std 1), AutoAugment khi train
TRAIN_TF = T.Compose([
    T.AutoAugment(T.AutoAugmentPolicy.IMAGENET, T.InterpolationMode.BILINEAR),
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
])
TEST_TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])

class CUBList(Dataset):
    def __init__(self, items, transform):
        self.items, self.transform = items, transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        with Image.open(p) as img:
            img = img.convert("RGB")
        return self.transform(img), y

def make_class_order(seed, n_classes):
    # giống CODA-P / APT / KAC: random.seed(seed) rồi shuffle 0..C-1
    order = list(range(n_classes))
    random.seed(seed)
    random.shuffle(order)
    return order


## 4. Model: ViT + APT prompt + head (cosine / KAC)

In [ ]:
# ---------------- ViT-B/16 (frozen) có hook cho Additive Prompt ----------------
class PatchEmbed(nn.Module):
    def __init__(self, dim=768, patch=16):
        super().__init__()
        self.proj = nn.Conv2d(3, dim, patch, patch)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class Attention(nn.Module):
    def __init__(self, dim=768, heads=12):
        super().__init__()
        self.heads, self.hd = heads, dim // heads
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x, prompt=None):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.heads, self.hd).permute(2, 0, 3, 1, 4).unbind(0)
        if prompt is not None:
            # APT: cộng prompt vào key/value của CLS token (vị trí 0), patch token giữ nguyên
            pk, pv = prompt                                   # (H, 1, hd)
            k = torch.cat([k[:, :, :1] + pk.to(k.dtype), k[:, :, 1:]], dim=2)
            v = torch.cat([v[:, :, :1] + pv.to(v.dtype), v[:, :, 1:]], dim=2)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(B, N, C))

class Mlp(nn.Module):
    def __init__(self, dim=768, hidden=3072):
        super().__init__()
        self.fc1, self.act, self.fc2 = nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim)
    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

class Block(nn.Module):
    def __init__(self, dim=768, heads=12):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, dim * 4)
    def forward(self, x, prompt=None):
        x = x + self.attn(self.norm1(x), prompt)
        return x + self.mlp(self.norm2(x))

class ViT(nn.Module):
    def __init__(self, dim=768, depth=12, heads=12):
        super().__init__()
        self.depth, self.heads = depth, heads
        self.patch_embed = PatchEmbed(dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 197, dim))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim, eps=1e-6)
    def forward(self, x, prompt_fn=None):
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(x.shape[0], -1, -1), x], dim=1) + self.pos_embed
        for i, blk in enumerate(self.blocks):
            x = blk(x, prompt_fn(i) if prompt_fn is not None else None)
        return self.norm(x)[:, 0]          # CLS feature

def load_pretrained_vit(name):
    try:
        ref = timm.create_model(name, pretrained=True, num_classes=0)
    except Exception as e:
        print(f"[warn] không tải được {name} ({e}); thử 'vit_base_patch16_224'")
        ref = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)
    sd = {k: v for k, v in ref.state_dict().items() if not k.startswith("head")}
    vit = ViT()
    missing, unexpected = vit.load_state_dict(sd, strict=False)
    assert not missing, f"Thiếu key: {missing}"
    if unexpected:
        print("[info] bỏ qua key:", unexpected)
    for p in vit.parameters():
        p.requires_grad_(False)
    return vit.eval(), ref.eval()

# ---------------- APT: một bộ prompt dùng chung + merge EMA (PPF) ----------------
class APTPrompt(nn.Module):
    def __init__(self, depth=12, dim=768, heads=12, ema_coeff=0.7):
        super().__init__()
        self.depth, self.heads, self.hd = depth, heads, dim // heads
        self.prompt_tokens = nn.Parameter(torch.zeros(depth * 2, dim))   # [k_0, v_0, k_1, v_1, ...]
        nn.init.trunc_normal_(self.prompt_tokens, std=0.02)
        self.register_buffer("merged", torch.zeros(depth * 2, dim))
        self.n_merged = 0
        self.ema_coeff = ema_coeff
        self.use_merged = False
    def get(self, l):
        P = self.merged if self.use_merged else self.prompt_tokens
        return P[2 * l].view(self.heads, 1, self.hd), P[2 * l + 1].view(self.heads, 1, self.hd)
    @torch.no_grad()
    def merge(self):
        # sau mỗi task: task 1 copy, các task sau merged = a*merged + (1-a)*current
        if self.n_merged == 0:
            self.merged.copy_(self.prompt_tokens)
        else:
            self.merged.mul_(self.ema_coeff).add_(self.prompt_tokens, alpha=1 - self.ema_coeff)
        self.n_merged += 1

# ---------------- Classifier heads ----------------
class CosineHead(nn.Module):
    """Head gốc của APT: LayerNorm (affine không được train) + linear với weight chuẩn hoá L2, không bias."""
    def __init__(self, dim, n_cls):
        super().__init__()
        self.norm = nn.LayerNorm(dim, elementwise_affine=False)
        self.fc = nn.Linear(dim, n_cls, bias=False)
    def forward(self, f):
        return self.norm(f) @ F.normalize(self.fc.weight, dim=1).t()

class KACHead(nn.Module):
    """KAC (Hu et al., CVPR 2025), theo models/kac_clf.py: LayerNorm -> Gaussian RBF -> spline linear (init 0)."""
    def __init__(self, dim, n_cls, num_grids=16, grid_range=(-2.0, 2.0)):
        super().__init__()
        gmin, gmax = grid_range
        self.layernorm = nn.LayerNorm(dim)                               # affine được train như code gốc
        self.register_buffer("grid", torch.linspace(gmin, gmax, num_grids))
        self.denominator = (gmax - gmin) / (num_grids - 1)
        self.spline_linear = nn.Linear(dim * num_grids, n_cls, bias=False)
        nn.init.zeros_(self.spline_linear.weight)
    def forward(self, f):
        x = self.layernorm(f)
        basis = torch.exp(-((x[..., None] - self.grid) / self.denominator) ** 2)   # (B, d, G)
        return self.spline_linear(basis.flatten(1))

class HybridHead(nn.Module):
    def __init__(self, dim, n_cls, num_grids=16, grid_range=(-2.0, 2.0), tau=2.0, alpha=1.5):
        super().__init__()
        self.cos_head = CosineHead(dim, n_cls)
        self.kac_head = KACHead(dim, n_cls, num_grids, grid_range)
        self.tau = tau
        self.alpha = alpha
    def forward(self, f):
        # Lấy feature sau LayerNorm của CosineHead
        out_norm = self.cos_head.norm(f)
        logit_cos = self.cos_head(f) # FIXED: Phải gọi qua hàm forward chuẩn của CosineHead để nó áp dụng L2 weight norm
        logit_kac = self.kac_head(f.detach())
        if self.training:
            return logit_cos, logit_kac
        else:
            return logit_cos, logit_kac, out_norm

class APTNet(nn.Module):
    def __init__(self, vit, n_cls, clf_type, cfg):
        super().__init__()
        self.vit = vit
        self.prompt = APTPrompt(vit.depth, 768, vit.heads, cfg["ema_coeff"])
        if clf_type == "linear":
            self.head = CosineHead(768, n_cls)
        elif clf_type == "kac":
            self.head = KACHead(768, n_cls, cfg["kac_num_grids"], cfg["kac_grid_range"])
        elif clf_type == "hybrid":
            self.head = HybridHead(768, n_cls, cfg["kac_num_grids"], cfg["kac_grid_range"], cfg["tau"], cfg["alpha"])
        else:
            raise ValueError(clf_type)
        self.use_amp = cfg["use_amp"]
    def forward(self, x):
        # train: prompt hiện tại; eval: prompt đã merge (nếu đã có)
        self.prompt.use_merged = (not self.training) and self.prompt.n_merged > 0
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp):
            f = self.vit(x, prompt_fn=self.prompt.get)
        with torch.autocast("cuda", enabled=False):
            return self.head(f.float())


Tải ViT-B/16 pretrained và kiểm tra khớp timm.

In [ ]:
VIT, REF = load_pretrained_vit(CFG["timm_name"])
VIT, REF = VIT.to(DEVICE), REF.to(DEVICE)

# Sanity check: ViT tự viết (không prompt) phải khớp timm (tắt TF32 khi so)
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
with torch.no_grad():
    x = torch.rand(2, 3, 224, 224, device=DEVICE)
    ref = REF.forward_features(x)[:, 0]
    rel = ((VIT(x) - ref).abs().max() / ref.abs().max()).item()
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
print(f"relative max diff với timm = {rel:.2e}")
assert rel < 1e-3, "ViT tự viết không khớp timm!"
del REF; torch.cuda.empty_cache()


## 5. Train / eval class-incremental

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=enabled)

@torch.no_grad()
def evaluate(model, test_items, n_seen_tasks, per, clf_type='hybrid', tau=2.0, alpha=1.5):
    hi = n_seen_tasks * per
    items = [it for it in test_items if it[1] < hi]
    loader = DataLoader(CUBList(items, TEST_TF), batch_size=128, shuffle=False,
                        num_workers=CFG["num_workers"], pin_memory=True)
    model.eval()
    all_logit_cos, all_logit_kac, all_f_norm, ys = [], [], [], []
    
    all_logits = []
    for x, y in loader:
        out = model(x.to(DEVICE, non_blocking=True))
        if clf_type == "hybrid":
            logit_cos, logit_kac, f_norm = out
            all_logit_cos.append(logit_cos.cpu())
            all_logit_kac.append(logit_kac.cpu())
            all_f_norm.append(f_norm.cpu())
        else:
            all_logits.append(out.cpu())
        ys.append(y)
        
    y = torch.cat(ys)
    task_of = y // per
    
    if clf_type == "hybrid":
        logit_cos = torch.cat(all_logit_cos)[:, :hi]
        logit_kac = torch.cat(all_logit_kac)[:, :hi]
        f_norm = torch.cat(all_f_norm)
        
        # 1. Effective Rank
        _, S, _ = torch.linalg.svd(f_norm.float(), full_matrices=False)
        S = S / S.sum()
        entropy = - (S * torch.log(S + 1e-9)).sum()
        eff_rank = torch.exp(entropy).item()
        
        # 2. Out of Grid
        u_out_of_grid = ((f_norm < -2.0) | (f_norm > 2.0)).float().mean().item() * 100
        
        # 3. Task-Aware Accuracy
        correct_ta = 0
        for j in range(n_seen_tasks):
            lo, hi_task = j * per, (j + 1) * per
            mask = (task_of == j)
            if mask.sum() > 0:
                prob_cos = F.softmax(logit_cos[mask][:, lo:hi_task] / tau, dim=1)
                prob_kac = F.softmax(logit_kac[mask][:, lo:hi_task], dim=1)
                pred_ta = (prob_cos + alpha * prob_kac).argmax(dim=1) + lo
                correct_ta += (pred_ta == y[mask]).sum().item()
        acc_ta = 100.0 * correct_ta / len(y)
        
        # 4. Standard CIL Accuracy
        prob_cos = F.softmax(logit_cos / tau, dim=1)
        prob_kac = F.softmax(logit_kac, dim=1)
        p = (prob_cos + alpha * prob_kac).argmax(dim=1)
        
        diag = {"eff_rank": eff_rank, "u_out_of_grid": u_out_of_grid, "acc_ta": acc_ta}
        
        if n_seen_tasks == CFG["n_tasks"]:
            print("\n[OFFLINE GRID SEARCH] Tối ưu hóa tau và alpha trên logits đã lưu...")
            best_acc, best_tau, best_alpha = 0, 0, 0
            for search_tau in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 8.0, 10.0]:
                for search_alpha in [0.1, 0.5, 1.0, 1.2, 1.5, 1.8, 2.0, 2.5, 3.0]:
                    p_c = F.softmax(logit_cos / search_tau, dim=1)
                    p_k = F.softmax(logit_kac, dim=1)
                    pred = (p_c + search_alpha * p_k).argmax(dim=1)
                    acc = 100.0 * (pred == y).float().mean().item()
                    if acc > best_acc:
                        best_acc, best_tau, best_alpha = acc, search_tau, search_alpha
            print(f"-> Tốt nhất: tau={best_tau}, alpha={best_alpha} => Accuracy: {best_acc:.2f}%")
            diag["best_grid_tau"] = best_tau
            diag["best_grid_alpha"] = best_alpha
            diag["best_grid_acc"] = best_acc
    else:
        # Standard evaluation for linear or kac
        logits = torch.cat(all_logits)[:, :hi]
        p = logits.argmax(dim=1)
        
        correct_ta = 0
        for j in range(n_seen_tasks):
            lo, hi_task = j * per, (j + 1) * per
            mask = (task_of == j)
            if mask.sum() > 0:
                pred_ta = logits[mask][:, lo:hi_task].argmax(dim=1) + lo
                correct_ta += (pred_ta == y[mask]).sum().item()
        acc_ta = 100.0 * correct_ta / len(y)
        diag = {"eff_rank": 0.0, "u_out_of_grid": 0.0, "acc_ta": acc_ta}

    accs = [100.0 * (p[task_of == j] == y[task_of == j]).float().mean().item() for j in range(n_seen_tasks)]
    overall = 100.0 * (p == y).float().mean().item()

    return accs, overall, diag

def summarize(R):
    Tn = R.shape[0]
    A = [float(np.mean(R[t, :t + 1])) for t in range(Tn)]
    fgt = float(np.mean([R[j, j] - R[Tn - 1, j] for j in range(Tn - 1)])) if Tn > 1 else 0.0
    return dict(A_t=A, last=A[-1], avg=float(np.mean(A)), forgetting=fgt)

def run_experiment(seed, clf_type):
    set_seed(seed)
    C, nT = CFG["n_classes"], CFG["n_tasks"]
    per = C // nT
    order = make_class_order(seed, C)
    remap = {c: i for i, c in enumerate(order)}           # task t = nhãn mới [t*per, (t+1)*per)
    tr = [(p, remap[y]) for p, y in TRAIN_ITEMS]
    te = [(p, remap[y]) for p, y in TEST_ITEMS]
    Tn = nT if CFG["max_tasks"] is None else min(nT, CFG["max_tasks"])

    model = APTNet(VIT, C, clf_type, CFG).to(DEVICE)
    n_train = sum(p.numel() for p in [model.prompt.prompt_tokens, *model.head.parameters()])
    print(f"\n######## {clf_type.upper()} | seed {seed} | trainable params: {n_train/1e6:.3f}M ########")

    R = np.zeros((Tn, Tn))
    log = []
    for t in range(Tn):
        lo, hi = t * per, (t + 1) * per
        loader = DataLoader(CUBList([it for it in tr if lo <= it[1] < hi], TRAIN_TF),
                            batch_size=CFG["batch_size"], shuffle=True, drop_last=True,
                            num_workers=CFG["num_workers"], pin_memory=True)
        # reset optimizer mỗi task (như APT/KAC); prompt dùng chung, tiếp tục từ task trước
        if clf_type == "hybrid":
            opt = torch.optim.Adam([
                {"params": [model.prompt.prompt_tokens], "lr": CFG["prompt_lr"]},
                {"params": list(model.head.cos_head.parameters()), "lr": CFG["head_lr"]["hybrid_cos"]},
                {"params": list(model.head.kac_head.parameters()), "lr": CFG["head_lr"]["hybrid_kac"]},
            ], betas=(0.9, 0.999), weight_decay=0.0)
        else:
            opt = torch.optim.Adam([
                {"params": [model.prompt.prompt_tokens], "lr": CFG["prompt_lr"]},
                {"params": list(model.head.parameters()), "lr": CFG["head_lr"][clf_type]},
            ], betas=(0.9, 0.999), weight_decay=0.0)
        base_lrs = [g["lr"] for g in opt.param_groups]
        scaler = make_scaler(CFG["use_amp"])
        E = CFG["epochs"]
        t0 = time.time()
        grads_this_task = []
        for ep in range(E):
            fac = math.cos(99 * math.pi * ep / (200 * max(E - 1, 1)))
            for g, b in zip(opt.param_groups, base_lrs):
                g["lr"] = b * fac
            model.train()
            tot, correct, n = 0.0, 0, 0
            ep_p_grad, ep_c_grad, steps = 0.0, 0.0, 0
            for bi, (x, y) in enumerate(loader):
                if CFG["max_train_batches"] and bi >= CFG["max_train_batches"]:
                    break
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                out = model(x)
                if clf_type == "hybrid":
                    logits_cos = out[0][:, lo:hi]
                    logits_kac = out[1][:, lo:hi]
                    loss = F.cross_entropy(logits_cos, y - lo) + F.cross_entropy(logits_kac, y - lo)
                    logits = logits_cos # For accuracy computation
                else:
                    logits = out[:, lo:hi]
                    loss = F.cross_entropy(logits, y - lo)
                
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                
                # --- GRADIENT TRACKING ---
                scaler.unscale_(opt)
                with torch.no_grad():
                    if hasattr(model.prompt, 'prompt_tokens') and model.prompt.prompt_tokens.grad is not None:
                        ep_p_grad += model.prompt.prompt_tokens.grad.norm().item()
                    c_grad = 0.0
                    for p in model.head.parameters():
                        if p.grad is not None:
                            c_grad += p.grad.norm().item()
                    ep_c_grad += c_grad
                steps += 1
                # -------------------------
                
                scaler.step(opt)
                scaler.update()
                tot += loss.item() * len(y); n += len(y)
                correct += (logits.argmax(1) == (y - lo)).sum().item()
                
            if steps > 0:
                grads_this_task.append({
                    "step": ep + 1,
                    "p_grad": round(ep_p_grad / steps, 4),
                    "c_grad": round(ep_c_grad / steps, 4)
                })
            
            if ep == 0 or ep == E - 1 or (ep + 1) % 6 == 0:
                print(f"  task {t+1} ep {ep+1:2d}/{E} | loss {tot/max(n,1):.3f} | train acc {100*correct/max(n,1):.1f}")
        model.prompt.merge()
        accs, overall, diag = evaluate(model, te, t + 1, per, clf_type=clf_type, tau=CFG["tau"], alpha=CFG["alpha"])
        R[t, :t + 1] = accs
        A_t = float(np.mean(accs))
        log.append(dict(task=t + 1, A_t=A_t, sample_acc=overall, train_time=time.time() - t0, accs=accs, grads=grads_this_task, diag=diag))
        print(f"→ sau task {t+1}: A_t = {A_t:.2f} | acc_ta = {diag['acc_ta']:.2f}% | rank = {diag['eff_rank']:.1f} | out_grid = {diag['u_out_of_grid']:.1f}%")

    res = summarize(R)
    
    formatted_metrics = {}
    for i_t, l_data in enumerate(log):
        formatted_metrics[str(i_t)] = {
            "accs": l_data["accs"],
            "grads": l_data["grads"]
        }
    
    res.update(R=R.tolist(), log=log, class_order=order, seed=seed, clf_type=clf_type, formatted_metrics=formatted_metrics)
    print(f"==> {clf_type} seed {seed}: Last {res['last']:.2f} | Avg {res['avg']:.2f} | Forgetting {res['forgetting']:.2f}")
    torch.save({"prompt_tokens": model.prompt.prompt_tokens.detach().cpu(),
                "merged": model.prompt.merged.cpu(),
                "head": {k: v.cpu() for k, v in model.head.state_dict().items()},
                "class_order": order},
               os.path.join(CFG["out_dir"], f"apt_{clf_type}_seed{seed}.pt"))
    del model, opt; torch.cuda.empty_cache()
    return res


## 6. Chạy
Kết quả được lưu vào `/kaggle/working/outputs/results.json` sau mỗi run (nếu session bị ngắt thì chạy lại sẽ bỏ qua run đã xong).

In [ ]:
RESULTS_PATH = os.path.join(CFG["out_dir"], "results.json")
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) and not DEBUG else {}

for n_tasks in [5, 10, 20, 40]:
    CFG["n_tasks"] = n_tasks
    c_per_t = CFG["n_classes"] // n_tasks
    for seed in CFG["seeds"]:
        for clf in CFG["clf_types"]:
            key = f"APT_{clf}_{n_tasks}T_{c_per_t}C"
            if key in RESULTS:
                print("bỏ qua (đã có):", key); continue
            print(f"\n{'='*50}\n BẮT ĐẦU: {key} \n{'='*50}")
            RESULTS[key] = run_experiment(seed, clf)
            json.dump(RESULTS, open(RESULTS_PATH, "w"), indent=1)   # lưu sau mỗi run


## 7. Kết quả

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rows = []
for k, r in RESULTS.items():
    rows.append(dict(method="APT" if r["clf_type"] == "linear" else ("APT + KAC" if r["clf_type"] == "kac" else "APT + Hybrid"),
                     seed=r["seed"], last=r["last"], avg=r["avg"], forgetting=r["forgetting"]))
df = pd.DataFrame(rows)
summary = df.groupby("method")[["last", "avg", "forgetting"]].agg(["mean", "std"]).round(2)
display(df.round(2)); display(summary)
summary.to_csv(os.path.join(CFG["out_dir"], "summary.csv"))

fig, axes = plt.subplots(1, 1 + len(CFG["clf_types"]), figsize=(5 * (1 + len(CFG["clf_types"])), 4))
for clf, name in [("linear", "APT"), ("kac", "APT + KAC"), ("hybrid", "APT + Hybrid")]:
    curves = [r["A_t"] for r in RESULTS.values() if r["clf_type"] == clf]
    if curves:
        m = np.mean(curves, 0)
        axes[0].plot(range(1, len(m) + 1), m, marker="o", label=name)
axes[0].set(xlabel="task", ylabel="A_t (%)", title="Accuracy sau mỗi task (CUB-200)")
axes[0].legend(); axes[0].grid(alpha=.3)
for ax, clf in zip(axes[1:], CFG["clf_types"]):
    rs = [r for r in RESULTS.values() if r["clf_type"] == clf]
    if not rs: continue
    R = np.array(rs[0]["R"]); R = np.where(np.tril(np.ones_like(R)) > 0, R, np.nan)
    im = ax.imshow(R, vmin=0, vmax=100, cmap="viridis")
    ax.set(title=f"R[t, j] – {clf} (seed {rs[0]['seed']})", xlabel="task được đánh giá j", ylabel="sau khi học task t")
    plt.colorbar(im, ax=ax, fraction=.046)
plt.tight_layout()
plt.savefig(os.path.join(CFG["out_dir"], "curves.png"), dpi=150)
plt.show()
